EXPLORE DATA

CHARGEMENT

In [15]:
import duckdb
import urllib.request
import zipfile
import os

# ── 1. Téléchargement et extraction du fichier ──────────────────────────────
url = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20260405-002321/valeursfoncieres-2025.txt.zip"
zip_path = "valeursfoncieres-2025.zip"
txt_path = "valeursfoncieres-2025.txt"

print("Téléchargement en cours...")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(".")
    txt_path = z.namelist()[0]  # récupère le nom exact du fichier extrait

print(f"Fichier extrait : {txt_path}")

# ── 2. Connexion DuckDB et chargement ────────────────────────────────────────
con = duckdb.connect("")

# Le fichier DVF est séparé par des "|" (pipe)
con.execute(f"""
    CREATE TABLE dvf AS
    SELECT * FROM read_csv_auto('{txt_path}',
        delim='|',
        header=True,
        ignore_errors=True
    )
""")

# ── 3. Nombre total de lignes ─────────────────────────────────────────────────
total = con.execute("SELECT COUNT(*) FROM dvf").fetchone()[0]
print(f"\n Nombre total de lignes : {total:,}")

# ── 4. Variables à conserver ─────────────────────────────────────────────────

cols = [row[0] for row in con.execute("DESCRIBE dvf").fetchall()]
print(f"\n Toutes les colonnes disponibles ({len(cols)}) :")
print(cols)

# Colonnes cibles (avec wildcard adresse_*)
cols_to_keep_explicit = [
    "Valeur fonciere", "Surface reelle bati",
    "Nombre pieces principales", "Type local","Nature mutation",
    "Code commune", "Date mutation","Code postal","Commune","Code departement","Code voie", "Voie"
]
cols_to_keep = cols_to_keep_explicit 

# Garder uniquement les colonnes qui existent réellement
cols_to_keep = [c for c in cols_to_keep if c in cols]
print(f"\n Colonnes conservées ({len(cols_to_keep)}) : {cols_to_keep}")

# ── 5. Valeurs manquantes par colonne ────────────────────────────────────────
print("\n Valeurs manquantes par colonne :")
for col in cols_to_keep:
    nulls = con.execute(f"SELECT COUNT(*) FROM dvf WHERE \"{col}\" IS NULL").fetchone()[0]
    pct = round(100 * nulls / total, 2)
    print(f"  {col:<35} : {nulls:>10,} manquantes  ({pct}%)")

# ── 6. Création de la table nettoyée (colonnes sélectionnées) ────────────────
select_clause = ", ".join([f'"{c}"' for c in cols_to_keep])
con.execute(f"""
    CREATE TABLE dvf_clean AS
    SELECT {select_clause}
    FROM dvf
""")

clean_count = con.execute("SELECT COUNT(*) FROM dvf_clean").fetchone()[0]
print(f"\n Table dvf_clean créée avec {clean_count:,} lignes et {len(cols_to_keep)} colonnes")

# ── 7. Nettoyage des valeurs manquantes ──────────────────────────────────────

# 1. Supprimer les lignes où valeur_fonciere est NULL
con.execute("""
    CREATE TABLE dvf_clean2 AS
    SELECT * FROM dvf_clean
    WHERE "Valeur fonciere" IS NOT NULL
""")

count_after = con.execute("SELECT COUNT(*) FROM dvf_clean2").fetchone()[0]
print(f" Lignes après suppression des NULL sur Valeur fonciere : {count_after:,}")

# 2. Remplacer les NULL par 0 pour les colonnes numériques
cols_numeric = [
    "Surface reelle bati",
    "Nombre pieces principales","Code postal"
]
for col in cols_numeric:
    con.execute(f"""
        UPDATE dvf_clean2
        SET "{col}" = 0
        WHERE "{col}" IS NULL
    """)
    print(f"  ✔ NULL remplacés par 0 dans : {col}")

# 3. Remplacer les NULL par 'NR' pour les colonnes texte
cols_text = [
    "Type local",
    "Code voie",
    "Voie"
]
for col in cols_text:
    con.execute(f"""
        UPDATE dvf_clean2
        SET "{col}" = 'NR'
        WHERE "{col}" IS NULL
    """)
    print(f"  ✔ NULL remplacés par 'NR' dans : {col}")

# ── 8. Vérification finale ───────────────────────────────────────────────────
print("\n Vérification après nettoyage :")
total_clean = con.execute("SELECT COUNT(*) FROM dvf_clean2").fetchone()[0]
print(f"  Nombre de lignes finales : {total_clean:,}")

for col in cols_numeric + cols_text + ["Valeur fonciere"]:
    nulls = con.execute(f'SELECT COUNT(*) FROM dvf_clean2 WHERE "{col}" IS NULL').fetchone()[0]
    print(f"  {col:<35} : {nulls} NULL restants")

# ── 9. Aperçu ────────────────────────────────────────────────────────────────
print("\n Aperçu des 5 premières lignes :")
print(con.execute("SELECT * FROM dvf_clean2 LIMIT 20").df().to_string())

Téléchargement en cours...
Fichier extrait : ValeursFoncieres-2025.txt


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Nombre total de lignes : 3,714,493

 Toutes les colonnes disponibles (43) :
['Identifiant de document', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Type de voie', 'Code voie', 'Voie', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain']

 Colonnes conservées (12) : ['Valeur fonciere', 'Surface reelle bati', 'Nombre pieces principales', 'Type local', 'Nature mutation', 'Co

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Table dvf_clean créée avec 3,714,493 lignes et 12 colonnes


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Lignes après suppression des NULL sur Valeur fonciere : 3,666,073
  ✔ NULL remplacés par 0 dans : Surface reelle bati
  ✔ NULL remplacés par 0 dans : Nombre pieces principales
  ✔ NULL remplacés par 0 dans : Code postal
  ✔ NULL remplacés par 'NR' dans : Type local
  ✔ NULL remplacés par 'NR' dans : Code voie
  ✔ NULL remplacés par 'NR' dans : Voie

 Vérification après nettoyage :
  Nombre de lignes finales : 3,666,073
  Surface reelle bati                 : 0 NULL restants
  Nombre pieces principales           : 0 NULL restants
  Code postal                         : 0 NULL restants
  Type local                          : 0 NULL restants
  Code voie                           : 0 NULL restants
  Voie                                : 0 NULL restants
  Valeur fonciere                     : 0 NULL restants

 Aperçu des 5 premières lignes :
   Valeur fonciere  Surface reelle bati  Nombre pieces principales   Type local Nature mutation  Code commune Date mutation  Code postal    Commune Co